Imports

In [2]:
%matplotlib inline
import pandas as pd
import numpy as np
import kagglehub
from umap import UMAP
from kagglehub import KaggleDatasetAdapter
from sentence_transformers import SentenceTransformer
from hdbscan import HDBSCAN
from bertopic import BERTopic
from bertopic.vectorizers import OnlineCountVectorizer
from langchain_google_genai import ChatGoogleGenerativeAI
from bertopic.representation import LangChain

/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Load Data

In [ ]:
df = pd.read_csv('Data/combined_timeline_data.csv')
df.head()

Preprocess Data

In [ ]:
initial_review_count = len(df)
df = df.dropna(subset=['Date','Source','Summary'])
# Robust date parsing: try pandas, fallback to dateutil for hard cases
def _parse_dates(series):
    parsed = pd.to_datetime(series, errors='coerce', infer_datetime_format=True)
    if parsed.isna().any():
        from dateutil import parser
        mask = parsed.isna()
        def _try(x):
            try:
                return parser.parse(x, dayfirst=False)
            except Exception:
                return pd.NaT
        parsed.loc[mask] = series[mask].apply(lambda x: _try(x) if isinstance(x, str) else pd.NaT)
    return pd.to_datetime(parsed, errors='coerce')

df['Date_parsed'] = _parse_dates(df['Date'].astype(str))
n_missing = int(df['Date_parsed'].isna().sum())
if n_missing:
    print(f"Warning: {n_missing} rows have unparseable dates and will be dropped.")
df = df.dropna(subset=['Date_parsed']).reset_index(drop=True)

# Normalize to ISO date string (YYYY-MM-DD) and create timestamps used by BERTopic
df['Date'] = df['Date_parsed'].dt.strftime('%Y-%m-%d')
docs = df["Summary"].tolist()
timestamps = pd.to_datetime(df["Date"]).dt.date

Prepare Models

In [ ]:
sentence_model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = sentence_model.encode(docs, show_progress_bar=True)

Train BERTopic

In [ ]:
print("Training BERTopic model...\n")


umap_model = UMAP(n_neighbors=15, n_components=10, metric='cosine')
hdbscan_model = HDBSCAN(min_cluster_size=10, metric='euclidean', cluster_selection_method='eom', prediction_data=True)
vectorizer_model = OnlineCountVectorizer(stop_words="english")


topic_model = BERTopic(
  embedding_model=sentence_model,
  umap_model=umap_model,
  hdbscan_model=hdbscan_model,
  vectorizer_model=vectorizer_model,
)


# Train BERTopic
topic_model = topic_model.fit(docs, embeddings)


print("\nModel training complete!")

Get Topic Information

In [ ]:
# Get topic information
topic_model.get_topic_info()

Visualize Topics

In [ ]:
topic_model.visualize_documents(docs, embeddings=embeddings)

Visualize Topic Similarity

In [ ]:
fig = topic_model.visualize_heatmap()
fig.show()

Hierarchichal Topic Visualization

In [ ]:
hierarchical_topics = topic_model.hierarchical_topics(docs)
fig = topic_model.visualize_hierarchy(hierarchical_topics=hierarchical_topics)
fig.show()

print("💡 This shows the hierarchical structure of topics")

Visualize Topics over Time

In [ ]:
topics_over_time = topic_model.topics_over_time(docs, timestamps)
topic_model.visualize_topics_over_time(topics_over_time)